# PyTorch — Tensors

**PyTorch** is het meest gebruikte framework voor deep learning in Python. De kern van PyTorch is de **Tensor**: een multidimensionale array die vrijwel identiek is aan een NumPy `ndarray`, maar met twee cruciale extra's:

1. **GPU-ondersteuning** — berekeningen kunnen op een grafische kaart worden uitgevoerd, wat deep learning 10–1000× sneller maakt
2. **Automatisch differentiëren** — PyTorch houdt bij hoe tensors worden berekend en kan automatisch afgeleiden berekenen (nodig voor backpropagation)

| | NumPy `ndarray` | PyTorch `Tensor` |
|---|---|---|
| Aanmaken | `np.array([1, 2, 3])` | `torch.tensor([1, 2, 3])` |
| Shape | `.shape` | `.shape` |
| Dtype | `.dtype` | `.dtype` |
| GPU | Nee | Ja (`.to("cuda")`) |
| Autograd | Nee | Ja (`requires_grad=True`) |
| Standaard dtype | `float64` | `float32` |

Als je NumPy kent, is de overstap naar PyTorch klein — de API is bewust gelijkaardig gehouden.

In [ ]:
import numpy as np
import torch

print(torch.__version__)

2.12.0+cu130


## Tensors aanmaken

De meeste aanmaakfuncties hebben een NumPy-tegenhanger.

In [2]:
# From a Python list
v = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
print(v)
print(type(v))

tensor([1., 2., 3., 4., 5.])
<class 'torch.Tensor'>


In [3]:
# 2D tensor (matrix): list of lists
M = torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(M)

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])


In [ ]:
# Equivalent of NumPy creation functions
print(torch.zeros(3, 4))  # np.zeros((3, 4))
print()
print(torch.ones(2, 3))  # np.ones((2, 3))
print()
print(torch.eye(4))  # np.eye(4)
print()
print(torch.arange(0, 10, 2))  # np.arange(0, 10, 2)

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])

tensor([[1., 1., 1.],
        [1., 1., 1.]])

tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]])

tensor([0, 2, 4, 6, 8])


In [ ]:
# Random tensors
print(torch.rand(3, 4))  # uniform [0, 1) — like np.random.random()
print()
print(torch.randn(3, 4))  # standard normal — like np.random.randn()
print()
print(torch.randint(0, 10, (2, 5)))  # integers — like np.random.randint()

tensor([[0.2763, 0.5821, 0.5514, 0.7456],
        [0.2867, 0.5624, 0.2280, 0.0236],
        [0.0494, 0.7816, 0.8884, 0.6346]])

tensor([[-0.8383, -0.7496, -0.8481,  1.5087],
        [-0.0327,  0.2113, -1.1090,  0.9906],
        [-0.6065, -0.5612, -0.0486,  0.5649]])

tensor([[4, 8, 2, 5, 0],
        [4, 2, 4, 7, 1]])


## Tensor inspecteren

Dezelfde drie attributen als bij NumPy, plus twee extra's: `dtype` en `device`.

In [6]:
t = torch.randn(3, 4)

print("shape: ", t.shape)  # (3, 4)
print("ndim:  ", t.ndim)  # 2
print("numel: ", t.numel())  # 12 — total number of elements (like np.size)
print("dtype: ", t.dtype)  # torch.float32
print("device:", t.device)  # cpu

shape:  torch.Size([3, 4])
ndim:   2
numel:  12
dtype:  torch.float32
device: cpu


## Datatypes (`dtype`)

Het standaard dtype in PyTorch is **`torch.float32`** — niet `float64` zoals in NumPy. Dit is bewust: de meeste GPU's zijn geoptimaliseerd voor 32-bit floats, en voor deep learning is de extra precisie van 64-bit zelden nodig.

In [7]:
a = torch.tensor([1.0, 2.0, 3.0])  # float32 (default)
b = torch.tensor([1, 2, 3])  # int64 (integers stay int64)
c = torch.tensor([1.0, 2.0], dtype=torch.float64)  # explicit float64
d = torch.tensor([True, False, True])  # bool

print(a.dtype, b.dtype, c.dtype, d.dtype)

torch.float32 torch.int64 torch.float64 torch.bool


In [8]:
# Convert dtype with .to() or .float() / .long() shorthand
b_float = b.to(torch.float32)  # int64 -> float32
a_long = a.long()  # float32 -> int64

print(b_float.dtype, a_long.dtype)

torch.float32 torch.int64


## NumPy-brug

PyTorch tensors en NumPy arrays kunnen naar elkaar worden geconverteerd. Belangrijk: ze **delen geheugen** zolang de tensor op de CPU staat en geen gradient bijhoudt.

In [9]:
# NumPy -> PyTorch
arr = np.array([1.0, 2.0, 3.0], dtype=np.float32)
t_from_np = torch.from_numpy(arr)

print("Tensor:", t_from_np)
print("dtype: ", t_from_np.dtype)

Tensor: tensor([1., 2., 3.])
dtype:  torch.float32


In [10]:
# PyTorch -> NumPy
t = torch.tensor([4.0, 5.0, 6.0])
arr_from_t = t.numpy()

print("Array:", arr_from_t)
print("type: ", type(arr_from_t))

Array: [4. 5. 6.]
type:  <class 'numpy.ndarray'>


In [11]:
# Shared memory: modifying the array also changes the tensor
arr = np.array([1.0, 2.0, 3.0], dtype=np.float32)
t = torch.from_numpy(arr)

arr[0] = 99.0  # modify the NumPy array...
print("Tensor after modifying array:", t)  # ...tensor changes too!

Tensor after modifying array: tensor([99.,  2.,  3.])


> Wil je een onafhankelijke kopie, gebruik dan `torch.tensor(arr)` (maakt altijd een kopie) in plaats van `torch.from_numpy(arr)`.

## Reshape

Net zoals NumPy's `reshape`, gebruikt PyTorch `.view()` of `.reshape()` om de vorm van een tensor te veranderen zonder data te kopiëren.

In [12]:
t = torch.arange(12)  # [0, 1, 2, ..., 11]
print(t)

matrix = t.reshape(3, 4)  # 3 rows, 4 cols
print(matrix)

flat = matrix.reshape(-1)  # -1 means "infer this dimension"
print(flat)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])


---

## Oefeningen

**Oefening 1** — Maak de volgende tensors aan en druk telkens `shape`, `ndim`, `dtype` en `device` af:
- Een 1D-tensor met de getallen 1 t/m 8 (gebruik `torch.arange`)
- Een 3×3-matrix van nullen met dtype `torch.float32`
- Een 4×4-eenheidstensor

In [ ]:
import torch

# your solution here

**Oefening 2** — Maak een NumPy-array `arr = np.linspace(0, 1, 10, dtype=np.float32)`. Converteer naar een PyTorch-tensor met `torch.from_numpy`. Verander dan `arr[0] = 99`. Wat zie je in de tensor? Leg uit waarom.

In [ ]:
import numpy as np
import torch

# your solution here

**Oefening 3** — Maak een tensor van vorm (2, 3, 4) gevuld met willekeurige waarden uit een standaardnormaalverdeling (`torch.randn`). Reshape die naar (6, 4) en daarna naar (24,). Verifieer het aantal elementen na elke stap met `.numel()`.

In [15]:
# your solution here

**Oefening 4** — Maak een integer-tensor `t = torch.tensor([1, 2, 3, 4, 5])`. Converteer hem naar:
- `float32` met `.to()`
- `float64` met `.to()`
- terug naar `int64` met de `.long()`-methode

Druk telkens het dtype af.

In [16]:
# your solution here

**Oefening 5** — Vergelijk NumPy en PyTorch naast elkaar: maak in beide bibliotheken:
- Een 5×5-matrix gevuld met het getal 3.14
- 1000 steekproeven uit een normaalverdeling met gemiddelde 2 en standaardafwijking 0.5

Druk telkens het gemiddelde en de standaardafwijking af. Zijn de API's gelijkaardig?

In [ ]:
import numpy as np
import torch

# your solution here